# Fixed Production QLoRA Training - Llama-2-13B

**CRITICAL FIXES APPLIED:**
- ✅ **Tokenizer Settings**: `clean_up_tokenization_spaces=False` to preserve SentencePiece markers
- ✅ **Data Preprocessing**: Metadata cleaning pipeline
- ✅ **Prompt Template**: Proper formatting with spacing
- ✅ **Uncertainty Training**: 15% examples teaching "I don't know"
- ✅ **Data Analysis**: Correlation matrices, quality metrics, distribution plots
- ✅ **Stable Dependencies**: Documented exact versions

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# CRITICAL: Use exact versions for reproducibility
!pip install -q \
    transformers==4.36.0 \
    peft==0.7.1 \
    bitsandbytes==0.41.3 \
    accelerate==0.25.0 \
    datasets==2.16.0 \
    tqdm \
    faiss-cpu \
    sentence-transformers \
    flash-attn --no-build-isolation \
    matplotlib seaborn pandas scikit-learn

# Document versions
import transformers, peft, bitsandbytes, torch
print('=' * 60)
print('DEPENDENCY VERSIONS')
print('=' * 60)
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')
print(f'bitsandbytes: {bitsandbytes.__version__}')
print(f'torch:        {torch.__version__}')
print('=' * 60)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import time

assert torch.cuda.is_available(), 'FATAL: No CUDA device found'

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print('=' * 60)
print('HARDWARE DIAGNOSTICS')
print('=' * 60)
print(f'  GPU:           {gpu_name}')
print(f'  VRAM:          {vram_gb:.1f} GB')
print(f'  CUDA Version:  {torch.version.cuda}')
print(f'  PyTorch:       {torch.__version__}')
print(f'  BF16 Support:  {torch.cuda.is_bf16_supported()}')

# Enable TF32 for H100
torch.backends.cuda.matmul.allow_tf32 = True
print('  TF32:          Enabled')
print('=' * 60)

## 1. Data Preprocessing & Cleaning

**CRITICAL FIXES:**
- Remove metadata artifacts (`|answered by|`, dates, references)
- Proper prompt template with spacing
- Add uncertainty examples (15%)

In [ ]:
import re
import random

def clean_text(text: str) -> str:
    """Remove web scraping artifacts that cause metadata leakage."""
    
    # Remove common metadata patterns
    patterns = [
        r'\[Reference:.*?\]',
        r'\|answered by\|.*?\|',
        r'\|date created\|.*?\|',
        r'\|last updated\|.*?\|',
        r'\|[Cc]omments\|.*',
        r'answered by:.*?(?=\n|$)',
        r'date created:.*?(?=\n|$)',
        r'last updated:.*?(?=\n|$)',
        r'\bComments:.*?(?=\n|$)',
        r'Source:.*?(?=\n|$)',
        r'Posted by:.*?(?=\n|$)',
        r'\d{1,2}/\d{1,2}/\d{2,4}',  # Dates
    ]
    
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.DOTALL)
    
    # Collapse multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*\|\s*$', '', text)  # Remove trailing pipes
    
    return text.strip()

def format_training_example(text: str, max_context_len: int = 800) -> str:
    """
    Format text into Q&A style with PROPER SPACING.
    
    CRITICAL: Space after 'Answer:' is essential for proper tokenization.
    """
    # Clean the text
    text = clean_text(text)
    
    if len(text) < 50:
        return None
    
    # Truncate to reasonable length
    text = text[:max_context_len]
    
    # Create Q&A format with EXPLICIT SPACING
    formatted = (
        f"Context: {text}\n\n"
        f"Question: Explain the main topic discussed.\n"
        f"Answer: "  # ← CRITICAL: Space after colon
    )
    
    return formatted

def create_uncertainty_example() -> str:
    """
    Generate examples teaching the model to express uncertainty.
    
    CRITICAL: 15% of training should be uncertainty examples.
    """
    uncertainty_templates = [
        (
            "Context: Information about photosynthesis in plants.\n\n"
            "Question: How does quantum computing work?\n"
            "Answer: The provided context discusses photosynthesis, not quantum computing. "
            "I don't have enough information to answer this question based on the given context."
        ),
        (
            "Context: Historical facts about World War II.\n\n"
            "Question: What are the latest AI research findings?\n"
            "Answer: The context focuses on World War II history. I cannot answer questions "
            "about recent AI research based on this information."
        ),
        (
            "Context: Basic mathematics concepts.\n\n"
            "Question: Explain the biochemical pathway of glycolysis.\n"
            "Answer: The context covers mathematics, not biochemistry. I don't have the necessary "
            "information to explain glycolysis from this context."
        ),
    ]
    
    return random.choice(uncertainty_templates)

print('Data cleaning functions loaded.')
print('Testing clean_text()...')
test_text = "Photosynthesis is... [Reference:[1]]|answered by|John Doe|date created|01/01/2020|"
print(f'Before: {test_text[:80]}')
print(f'After:  {clean_text(test_text)}')

## 2. Data Analysis & Visualization

Analyze dataset quality before training

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print('Loading sample for analysis...')
raw_dataset = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    split='train',
    streaming=True
)

# Sample 5000 examples for analysis
ANALYSIS_SAMPLES = 5000
sample_data = []

print(f'Sampling {ANALYSIS_SAMPLES:,} examples...')
for i, row in enumerate(raw_dataset):
    if i >= ANALYSIS_SAMPLES:
        break
    
    text = row['text']
    cleaned = clean_text(text)
    
    sample_data.append({
        'original_length': len(text),
        'cleaned_length': len(cleaned),
        'chars_removed': len(text) - len(cleaned),
        'has_metadata': int(len(text) > len(cleaned)),
        'word_count': len(text.split()),
        'avg_word_length': np.mean([len(w) for w in text.split()]) if text.split() else 0,
    })

df = pd.DataFrame(sample_data)
print('\nDataset Statistics:')
print('=' * 60)
print(df.describe())
print(f'\nMetadata found in: {df["has_metadata"].sum():,} / {len(df):,} samples '
      f'({100 * df["has_metadata"].mean():.1f}%)')

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Dataset Quality Analysis', fontsize=16)

# 1. Length Distribution
axes[0, 0].hist(df['cleaned_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Text Length Distribution (After Cleaning)')
axes[0, 0].set_xlabel('Character Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['cleaned_length'].median(), color='r', linestyle='--', label='Median')
axes[0, 0].legend()

# 2. Metadata Impact
axes[0, 1].hist(df['chars_removed'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Characters Removed by Cleaning')
axes[0, 1].set_xlabel('Characters Removed')
axes[0, 1].set_ylabel('Frequency')

# 3. Word Count Distribution
axes[0, 2].hist(df['word_count'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 2].set_title('Word Count Distribution')
axes[0, 2].set_xlabel('Word Count')
axes[0, 2].set_ylabel('Frequency')

# 4. Correlation Matrix
corr = df[['original_length', 'cleaned_length', 'chars_removed', 'word_count']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 0], square=True)
axes[1, 0].set_title('Feature Correlation Matrix')

# 5. Metadata Presence
metadata_counts = df['has_metadata'].value_counts()
axes[1, 1].bar(['Clean', 'Has Metadata'], 
               [metadata_counts.get(0, 0), metadata_counts.get(1, 0)],
               color=['green', 'red'], alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Metadata Presence')
axes[1, 1].set_ylabel('Sample Count')

# 6. Average Word Length
axes[1, 2].hist(df['avg_word_length'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 2].set_title('Average Word Length Distribution')
axes[1, 2].set_xlabel('Avg Word Length (chars)')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].axvline(df['avg_word_length'].median(), color='r', linestyle='--', label='Median')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/fineweb_edu_llama2_13b/data_analysis.png', dpi=300, bbox_inches='tight')
print('\nVisualization saved to data_analysis.png')
plt.show()

# Quality Metrics Summary
print('\n' + '=' * 60)
print('QUALITY METRICS')
print('=' * 60)
print(f'Avg text length:        {df["cleaned_length"].mean():.0f} chars')
print(f'Median text length:     {df["cleaned_length"].median():.0f} chars')
print(f'Avg words per sample:   {df["word_count"].mean():.0f}')
print(f'Avg word length:        {df["avg_word_length"].mean():.1f} chars')
print(f'Metadata removal rate:  {100 * df["has_metadata"].mean():.1f}%')
print(f'Avg chars cleaned:      {df["chars_removed"].mean():.0f}')
print('=' * 60)

## 3. Loading Model (With Fixed Tokenizer Settings)

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from tqdm import tqdm

MODEL_NAME = 'NousResearch/Llama-2-13b-hf'
MAX_LENGTH = 1024

print(f'Loading {MODEL_NAME} in 4-bit NF4...')

# CRITICAL FIX: Proper tokenizer settings to preserve spacing
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,  # ← CRITICAL: Preserves SentencePiece ▁ markers
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_LENGTH

# Verify tokenization preserves spacing markers
test_tokens = tokenizer.tokenize("The model outputs text")
print(f'\nTokenization Test:')
print(f'  Input: "The model outputs text"')
print(f'  Tokens: {test_tokens[:5]}')
if any('▁' in t for t in test_tokens):
    print('  ✅ SentencePiece markers preserved correctly')
else:
    print('  ⚠️  WARNING: SentencePiece markers may be missing')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

print('Base model loaded.')

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

model = get_peft_model(model, peft_config)

trainable, total = 0, 0
for p in model.parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print('=' * 60)
print('LORA CONFIGURATION')
print('=' * 60)
print(f'  Total params:     {total:,}')
print(f'  Trainable params: {trainable:,}')
print(f'  Trainable %:      {100 * trainable / total:.2f}%')
print('=' * 60)

## 4. Preparing Training Data (With Preprocessing & Uncertainty Examples)

In [ ]:
print('Initializing dataset stream with preprocessing...')

# Reload dataset for training
raw_dataset = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    split='train',
    streaming=True
)

shuffled_dataset = raw_dataset.shuffle(seed=42, buffer_size=10000)

def tokenize_with_preprocessing(examples):
    """
    CRITICAL FIX: Apply preprocessing and proper formatting.
    
    15% uncertainty examples are injected here.
    """
    processed_texts = []
    
    for text in examples['text']:
        # 15% chance of uncertainty example
        if random.random() < 0.15:
            processed_texts.append(create_uncertainty_example())
        else:
            # Clean and format regular example
            formatted = format_training_example(text)
            if formatted:
                processed_texts.append(formatted)
            else:
                # Fallback to uncertainty if text too short
                processed_texts.append(create_uncertainty_example())
    
    # Tokenize with proper settings
    return tokenizer(
        processed_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False  # Dynamic padding handled by data collator
    )

print('Configuring tokenization with preprocessing...')

# Detect columns to remove
try:
    sample = next(iter(shuffled_dataset))
    remove_cols = list(sample.keys())
    print(f'Removing columns: {remove_cols}')
except:
    remove_cols = ['text', 'id', 'url', 'dump', 'segment', 'timestamp']

tokenized_dataset = shuffled_dataset.map(
    tokenize_with_preprocessing,
    batched=True,
    batch_size=32,
    remove_columns=remove_cols
)

print('Dataset stream ready with:')
print('  ✅ Metadata cleaning')
print('  ✅ Proper prompt formatting')
print('  ✅ 15% uncertainty examples')
print('  ✅ Preserved SentencePiece markers')

## 5. Training Configuration

In [ ]:
output_dir = '/content/drive/MyDrive/fineweb_edu_llama2_13b_fixed/checkpoints'
os.makedirs(output_dir, exist_ok=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=150,
    max_steps=5000,
    bf16=True,
    optim='adamw_bnb_8bit',
    gradient_checkpointing=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=3,
    report_to='none',
    remove_unused_columns=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True
)

print('Training configuration ready.')

In [ ]:
from transformers import TrainerCallback
import gc

class MemoryCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[MemoryCallback()]
)

print('Trainer ready.')

## 6. Training

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(output_dir)
train_start = time.time()

try:
    if last_checkpoint:
        print(f'Resuming from: {last_checkpoint}')
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print('Starting training...')
        trainer.train()
    
    elapsed = (time.time() - train_start) / 60
    print(f'\n✅ Training complete: {elapsed:.1f} minutes')
    
except Exception as e:
    print(f'\n❌ Training failed: {e}')
    torch.cuda.empty_cache()

In [ ]:
final_model_dir = '/content/drive/MyDrive/fineweb_edu_llama2_13b_fixed/final_model'
print(f'Saving model to: {final_model_dir}')
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

# Save training metadata
import json
metadata = {
    'model': MODEL_NAME,
    'transformers_version': transformers.__version__,
    'peft_version': peft.__version__,
    'torch_version': torch.__version__,
    'max_length': MAX_LENGTH,
    'preprocessing': 'metadata_cleaning + uncertainty_injection',
    'uncertainty_rate': 0.15,
    'tokenizer_settings': {
        'clean_up_tokenization_spaces': False,
        'preserves_sentencepiece_markers': True
    }
}
with open(f'{final_model_dir}/training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Model and metadata saved.')

## 7. Build Clean RAG Index

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

RAG_SAMPLES = 100_000
RAG_DIR = '/content/drive/MyDrive/fineweb_edu_llama2_13b_fixed/rag_index'
os.makedirs(RAG_DIR, exist_ok=True)

# Reload dataset for RAG
rag_dataset = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    split='train',
    streaming=True
)
rag_stream = rag_dataset.take(RAG_SAMPLES)

passages = []
print(f'Extracting and cleaning {RAG_SAMPLES:,} passages...')
for row in tqdm(rag_stream, total=RAG_SAMPLES):
    text = row['text'].strip()
    
    # CRITICAL FIX: Clean metadata before chunking
    text = clean_text(text)
    
    # Chunk into passages
    for i in range(0, len(text), 500):
        chunk = text[i:i + 500].strip()
        if len(chunk) > 50:
            passages.append(chunk)

print(f'Encoding {len(passages):,} clean passages...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedder.encode(
    passages,
    show_progress_bar=True,
    batch_size=256,
    convert_to_numpy=True
)

# Build FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
faiss.normalize_L2(embeddings)
index.add(embeddings)

# Save
faiss.write_index(index, os.path.join(RAG_DIR, 'faiss_index.bin'))
np.save(os.path.join(RAG_DIR, 'passages.npy'), np.array(passages, dtype=object))

print(f'\n✅ Clean RAG index saved:')
print(f'   Passages: {len(passages):,}')
print(f'   Dimension: {embeddings.shape[1]}')
print(f'   Location: {RAG_DIR}')

## Training Complete!

### Key Fixes Applied:
1. ✅ Tokenizer: `clean_up_tokenization_spaces=False` (preserves ▁ markers)
2. ✅ Data Preprocessing: Metadata cleaning pipeline
3. ✅ Prompt Formatting: Proper spacing in templates
4. ✅ Uncertainty Training: 15% examples teaching "I don't know"
5. ✅ Data Analysis: Correlation matrices and quality metrics
6. ✅ Clean RAG Index: Metadata removed from passages
7. ✅ Version Control: Exact dependency versions documented

### Next Steps:
1. Test the model with `chat_llm.py`
2. Verify spacing is correct in outputs
3. Check uncertainty responses on off-topic questions
4. Compare to baseline model